In [1]:
#  importing necessary libraries
import json
import random
import dotenv
from sql_matches import sql_matches

# Loading the .env file to load environment variables, if any
dotenv.load_dotenv()

True

In [2]:
# If you dont have access to the dataset, this is how you can download it

"""
pip install datasets
python -c "
from datasets import load_dataset
import json
ds = load_dataset('b-mc2/sql-create-context', split='train')
with open('sql_create_context_v4.json', 'w') as f:
    json.dump([dict(row) for row in ds], f)
print(f'Saved {len(ds)} examples')
"
"""

'\npip install datasets\npython -c "\nfrom datasets import load_dataset\nimport json\nds = load_dataset(\'b-mc2/sql-create-context\', split=\'train\')\nwith open(\'sql_create_context_v4.json\', \'w\') as f:\n    json.dump([dict(row) for row in ds], f)\nprint(f\'Saved {len(ds)} examples\')\n"\n'

In [3]:
#  Loading and exploring the dataset
import random
with open("sql_create_context_v4.json") as f:
    data = json.load(f)

print(f"Total examples: {len(data)}")
for i in range(3):
    print(f"\nExample {i+1}:")
    sample = random.choice(data)
    print(f"  Question: {sample['question']}")
    print(f"  Context:  {sample['context'][:120]}...")
    print(f"  Answer:   {sample['answer']}")


Total examples: 78577

Example 1:
  Question: What is the population density of mongmong-toto-maite?
  Context:  CREATE TABLE table_2588674_1 (pop_density INTEGER, village VARCHAR)...
  Answer:   SELECT MIN(pop_density) FROM table_2588674_1 WHERE village = "Mongmong-Toto-Maite"

Example 2:
  Question: How many districts have an incumbent first elected in 1944?
  Context:  CREATE TABLE table_1342198_6 (district VARCHAR, first_elected VARCHAR)...
  Answer:   SELECT COUNT(district) FROM table_1342198_6 WHERE first_elected = 1944

Example 3:
  Question: Which Line has a Platform of 3?
  Context:  CREATE TABLE table_name_62 (line VARCHAR, platform VARCHAR)...
  Answer:   SELECT line FROM table_name_62 WHERE platform = "3"


In [4]:
#  Splitting the dataset into training and test sets

NUM_TEST_EXAMPLES = 200  # Held-out for evaluation; all remaining data used for training
random.seed(42)  # Set the seed for reproducibility
random.shuffle(data)  # Shuffle the data to ensure randomness 
test_data = data[:NUM_TEST_EXAMPLES]
train_data = data[NUM_TEST_EXAMPLES:]
print(f"Training examples: {len(train_data)} (all except evaluation)")
print(f"Test examples: {len(test_data)}")

Training examples: 78377 (all except evaluation)
Test examples: 200


In [5]:
# Prompt format 

prompt_format = '''Table schema:
{schema}
Question: {question}
SQL: {sql}'''

Step 3: Evaluating the base model

In [6]:
from tinker import types

def sample_from_model(sampling_client, tokenizer, context: str, question: str) -> str:
    """Generate SQL from the model given schema and question."""
    prompt = f"""Table schema:
{context}
Question: {question}
SQL: """
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    model_input = types.ModelInput.from_ints(tokens=prompt_tokens)
    params = types.SamplingParams(max_tokens=150, temperature=0.0, stop=["\n\n", "Question:"])
    result = sampling_client.sample(prompt=model_input, sampling_params=params, num_samples=1).result()
    if result.sequences:
        return tokenizer.decode(result.sequences[0].tokens).strip()
    return ""

def eval_one(sampling_client, tokenizer, ex: dict) -> bool:
    """Evaluate one example: generate SQL, then check if it matches expected."""
    sql = sample_from_model(sampling_client, tokenizer, ex["context"], ex["question"])
    return sql_matches(sql, ex["answer"], schema=ex["context"])

def evaluate_test_set(sampling_client, tokenizer, test_data: list) -> float:
    """Compute accuracy = fraction of test examples where generated SQL matches expected."""
    correct = sum(1 for ex in test_data if eval_one(sampling_client, tokenizer, ex))
    return correct / len(test_data)

In [7]:
import tinker

service_client = tinker.ServiceClient()
base_model = "meta-llama/Llama-3.2-1B"
training_client = service_client.create_lora_training_client(base_model=base_model)
tokenizer = training_client.get_tokenizer()

print("\n--- Evaluating Base Model on 200 Test Questions ---")
base_sampling_client = training_client.save_weights_and_get_sampling_client(
    name="base-model"
)
base_accuracy = evaluate_test_set(
    base_sampling_client, tokenizer, test_data
)
print(f"Base model accuracy: {base_accuracy:.2%} ({int(base_accuracy * 200)}/200)")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Evaluating Base Model on 200 Test Questions ---
Base model accuracy: 47.00% (94/200)


In [8]:
import numpy as np

# Preparing the training data for Tinker
def format_prompt(example: dict) -> tuple[str, str]:
    """Format example as prompt and completion for text-to-SQL."""
    prompt = f"""Table schema:
{example['context']}
Question: {example['question']}
SQL: """
    completion = example["answer"]
    return prompt, completion

def process_example(example: dict, tokenizer) -> types.Datum:
    """Convert a (question, context, answer) example into a Tinker Datum."""
    prompt, completion = format_prompt(example)

    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    prompt_weights = [0.0] * len(prompt_tokens)

    # Add space before completion, end with \n\n so the model learns to stop
    completion_str = f" {completion}\n\n"
    completion_tokens = tokenizer.encode(completion_str, add_special_tokens=False)
    completion_weights = [1.0] * len(completion_tokens)

    tokens = prompt_tokens + completion_tokens
    weights = prompt_weights + completion_weights

    # Next-token prediction: input is tokens[:-1], target is tokens[1:]
    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    weights = weights[1:]

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens),
        loss_fn_inputs={ 
            "target_tokens": np.array(target_tokens, dtype=np.int64),
            "weights": np.array(weights, dtype=np.float32),
        },
    )

In [9]:
# Preparing the training data
processed_train = [
    process_example(ex, tokenizer) for ex in train_data
]
random.shuffle(processed_train)

In [10]:
# Training loop with Tinker

from tinker import types

NUM_EPOCHS = 1
BATCH_SIZE = 256
LEARNING_RATE = 5e-4  # Tinker-recommended for Llama-3.2-1B with LoRA

step = 0
loss_history = []
for epoch in range(NUM_EPOCHS):
    random.shuffle(processed_train)
    for batch_idx in range(0, len(processed_train), BATCH_SIZE):
        batch = processed_train[batch_idx : batch_idx + BATCH_SIZE]
        if len(batch) == 0:
            break

        fwdbwd_future = training_client.forward_backward(batch, "cross_entropy")
        optim_future = training_client.optim_step(
            types.AdamParams(learning_rate=LEARNING_RATE)
        )

        fwdbwd_result = fwdbwd_future.result()
        optim_result = optim_future.result()

        # Compute loss (weighted cross-entropy over completion tokens only)
        to_arr = lambda x: x.to_numpy() if hasattr(x, "to_numpy") else np.array(x.tolist())
        logprobs = np.concatenate([to_arr(o["logprobs"]) for o in fwdbwd_result.loss_fn_outputs])
        weights = np.concatenate([to_arr(d.loss_fn_inputs["weights"]) for d in batch])
        loss = float(-np.dot(logprobs, weights) / (weights.sum() + 1e-8))

        step += 1
        if step % 100 == 0 or batch_idx + BATCH_SIZE >= len(processed_train):
            print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, update {step}, loss: {loss:.4f}")
            loss_history.append(loss)

Epoch 1/1, update 100, loss: 0.0488
Epoch 1/1, update 200, loss: 0.0306
Epoch 1/1, update 300, loss: 0.0391
Epoch 1/1, update 307, loss: 0.0288


In [ ]:
#  Evaluating the fine-tuned model on the test set

print("\n--- Evaluating Fine-Tuned Model on 200 Test Questions ---")
fine_tuned_sampling_client = training_client.save_weights_and_get_sampling_client(
    name="LoRA-fine-tuned-model"
)
fine_tuned_accuracy = evaluate_test_set(
    fine_tuned_sampling_client, tokenizer, test_data
)
print(f"Fine-tuned model accuracy: {fine_tuned_accuracy:.2%} ({int(fine_tuned_accuracy * 200)}/200)")


--- Evaluating Fine-Tuned Model on 200 Test Questions ---
Fine-tuned model accuracy: 89.50% (179/200)


In [13]:
#  Out of context examples

extra_examples = [
    {
        "level": "Easy (single table, simple WHERE)",
        "context": "CREATE TABLE employees (id INTEGER, name VARCHAR, salary REAL, department VARCHAR)",
        "question": "What are the names of employees in the engineering department?"
    },
    {
        "level": "Easy (single table, simple WHERE)",
        "context": "CREATE TABLE products (id INTEGER, name VARCHAR, price REAL, category VARCHAR)",
        "question": "How many products cost more than 50 dollars?"
    },
    {
        "level": "Medium (aggregation, ORDER BY)",
        "context": "CREATE TABLE students (id INTEGER, name VARCHAR, score INTEGER, class VARCHAR)",
        "question": "What is the highest score in the science class?"
    },
    {
        "level": "Medium (aggregation, ORDER BY)",
        "context": "CREATE TABLE orders (id INTEGER, customer VARCHAR, amount REAL, date VARCHAR)",
        "question": "List the top 3 customers by total order amount."
    },
    {
        "level": "Hard (JOIN, GROUP BY)",
        "context": "CREATE TABLE courses (id INTEGER, name VARCHAR, department VARCHAR); CREATE TABLE enrollments (student_id INTEGER, course_id INTEGER, grade VARCHAR)",
        "question": "How many students are enrolled in each department?"
    }
]

In [16]:
for i, ex in enumerate(extra_examples):
    print(f"\nExample {i+1} ({ex['level']}):")
    print(f"  Question: {ex['question']}")
    print(f"  Context:  {ex['context']}")

    ft_answer = sample_from_model(fine_tuned_sampling_client, tokenizer, ex["context"], ex["question"])
    print(f"  Fine-tuned Model-generated SQL: {ft_answer}")

    base_answer = sample_from_model(base_sampling_client, tokenizer, ex["context"], ex["question"])
    print(f"  Base Model-generated SQL: {base_answer}")


Example 1 (Easy (single table, simple WHERE)):
  Question: What are the names of employees in the engineering department?
  Context:  CREATE TABLE employees (id INTEGER, name VARCHAR, salary REAL, department VARCHAR)
  Fine-tuned Model-generated SQL: SELECT id, name FROM employees WHERE department = 'Engineering'
  Base Model-generated SQL: SELECT name FROM employees WHERE department = 'engineering'
Answer:  SELECT name FROM employees WHERE department = 'engineering'
Explanation:  The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the results of a query. The WHERE clause is used to filter the re